# decoder_only
## 与encoder-decoder架构不同点：
### 1.不含交叉注意力，而只有多头注意力
### 2.原本不含掩码的交叉注意力部分现在要加上掩码
### 3.原本输入encoder端的数据现在直接输入decoder端

In [91]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import math
from tqdm import tqdm

In [92]:
#hyper_parameters
num_heads=8
d_model=512
#模型内部使用的三个超参数，对外只有d_model
d_k=64
d_v=64 #保证d_v*num_head=d_model
d_ff=2048
lr=1e-4
epochs=1001
device='cuda:0'

## 1.decoder_only的注意力默认带掩码

In [93]:
def Attention(Q,K,V,mask=True):
#加上casule mask:
    B,H,N,_=Q.shape
    if mask==True:#decoder中有掩码，encoder无掩码
        mask=torch.tril(torch.ones(B,H,N,N))#tril生成下三角
        #再将所有0位置替换为布尔无穷小
        mask=mask.masked_fill(mask == 0, float('-inf'))
        #再将所有1替换为布尔0
        mask=mask.masked_fill(mask==1,0.0)
        mask=mask.to(Q.device)
        alpha=torch.softmax((Q@K.transpose(-2,-1))/(d_k**0.5)+mask,dim=-1)
    else:
        alpha=torch.softmax((Q@K.transpose(-2,-1))/(d_k**0.5),dim=-1)
    h=alpha@V
    return h

## 2.位置编码

In [94]:

class PositionalEmbed(nn.Module):
    def __init__(self,d_model,pos,base=10000,dropout=0.1):#embedding 后的x维度恰好是d_model，pos为序列最大长度（也就是每句tokens长度）
        super().__init__()
        self.dropout=nn.Dropout(p=dropout)#Dropout模块
        
        #position向量：
        p=torch.arange(0,pos,dtype=torch.float) #保证是float向量
        
        #角度向量:先生成等差的指数部分，再整体exp，否则无法直接生成
        factor=torch.arange(0,(d_model+1)//2,dtype=torch.float)
        factor=(-2/d_model*math.log(base))*factor
        factor=torch.exp(factor)

        angle=torch.matmul(p.unsqueeze(1),factor.unsqueeze(0))#torch的一维向量没有行列概念，所有一维向量相乘都只有内积，所以需要使用unsqueeze函数
        #（pos,）是一维，pytorch中一维向量没有行列之分，（pos,1）就是二维了（列矩阵）
        #matmul对一维向量执行点积，对二维向量执行矩阵乘法
        #可以使用 .reshape(pos, 1) 或 .view(pos, 1) 将一维张量 (pos,) 转换为二维列向量 (pos, 1)。这两种方法与 .unsqueeze(1) 的效果等价，都是增加一个维度。
        
        pe=torch.zeros(pos,d_model)
        pe[:,0::2]=torch.sin(angle)#0::2表示从0开始每隔两个取一个    #对整个e做出预修改然后取用
        if d_model % 2 == 0:
            pe[:, 1::2] = torch.cos(angle)
        else:
            pe[:, 1::2] = torch.cos(angle[:, :d_model//2]) 

        self.register_buffer('pe',pe)#将init中的局部变量e变为全局

        
    def forward(self,x):#forward函数负责将计算好的位置编码加到embedding(x)上,
                          #因为nn.Module机制规定只有forward函数里的操作才能参与反向传播，追踪梯度
        x=x+self.pe[:x.size(1)]#为了追踪这个加法的梯度
        x=self.dropout(x)
        return x   


# 3.embedding类：防止每次都重新实例化，写为类之后在后面的类中实例化，每次实例化一次即可

In [95]:
class Embedding(nn.Module):
    def __init__(self,num_embeddings,d_model):
        super().__init__()
        self.embedding=nn.Embedding(num_embeddings=num_embeddings,embedding_dim=d_model) 
    def forward(self,x):
        x=self.embedding(x)
        return x

## 4.不含交叉注意力的多头注意力

In [96]:
class Multihead(nn.Module):
    def __init__(self,d_model,d_k,d_v):
        super().__init__() 
        

        #多头注意力：[W1|W2|W3|W4|...]
        self.w_q=nn.Linear(d_model,d_k*num_heads) 
        self.w_k=nn.Linear(d_model,d_k*num_heads)
        self.w_v=nn.Linear(d_model,d_v*num_heads)  
        self.w0=nn.Linear(d_v*num_heads,d_model)
        #self.output_proj=nn.Linear(d_model,vocab_size+1)#transformer中不需要映射到词表大小，只需要保持d_model
    
    
    def forward(self,x,mask=True):#有可能是交叉注意力模块
        
        
        Q=self.w_q(x)
        K=self.w_k(x)
        V=self.w_v(x)
        #先计算后拆分
        Bq,Lq,numxKq=Q.shape#K与Q形状完全一致
        Bv,Lv,numxKv=V.shape
        Q=Q.reshape(Bq,Lq,num_heads,numxKq//num_heads)#reshape都是重排元素，所以只能拆为相邻元素
        K=K.reshape(Bq,Lq,num_heads,numxKq//num_heads)
        V=V.reshape(Bv,Lv,num_heads,numxKv//num_heads)
        Q = Q.transpose(1, 2)  # (B, L,num_heads, d_k) -> (B, num_heads, L, d_k)   转置第1，2维度
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)
        h_fin = Attention(Q, K, V)  # 输出 (B, Heads, L, d_v)
        h_fin = h_fin.transpose(1,2)  # (B, L, H, d_v)
        h_fin=h_fin.flatten(start_dim=2)  #(B,L,H*d_v)
        h_fin=self.w0(h_fin)


        #logits = self.output_proj(h_fin)  #出错，输出应该还是d_model

        return h_fin





## 6.decoder基础块

In [97]:
class Decoder_block(nn.Module):
    def __init__(self,d_model,d_k,d_v,d_ff=2048):
        super().__init__()
        self.multihead1=Multihead(d_model,d_k,d_v)
        self.multihead2=Multihead(d_model,d_k,d_v) #如果不写开就是参数复用
        self.LN1=nn.LayerNorm(d_model)
        self.LN2=nn.LayerNorm(d_model)#如果都用一套layernorm的话，参数复用
        self.LN3=nn.LayerNorm(d_model)
        self.dropout=nn.Dropout(p=0.1)
        self.ffn=nn.Sequential(
                nn.Linear(num_heads*d_v,d_ff), # num_heads*d_v=d_model
                nn.ReLU(),
                nn.Linear(d_ff,d_model)  #将输出维度投影回d_model维度，给下一个块使用
        )

    def forward(self,x):
        y1=self.multihead1(x,mask=True)
        y1=self.dropout(y1)
        x1=x+y1
        x2=self.LN1(x1)
        #无交叉注意力：每一个decoder块都接收最后一个encoder块的输出作为交叉注意力QK来源
        y2=self.multihead2(x2,mask=True)#改交叉注意力不用掩码为需要
        y2=self.dropout(y2)
        x2=x2+y2

        x3=self.LN2(x2)
        
        y3=self.ffn(x3)
        y3=self.dropout(y3)
        x4=x3+y3
        x4=self.LN3(x4)
        return x4
        

## 7.decoder

In [98]:
class Decoder(nn.Module):
    def __init__(self,d_model,d_k,d_v,d_ff=2048,block_num=6):
        super().__init__()
        #多层block堆叠写法：
       
        self.decoder=nn.ModuleList([
            Decoder_block(d_model,d_k,d_v) for _ in range(block_num)
            ])#使用列表动态规定encoder_block块数量
    def forward(self,x):
        for block in self.decoder:
            x=block(x) #可以实现串行
        return x 

## 8.输出处理类（同样写成类是为了防止每次重置）

In [99]:
class Output_operation(nn.Module):
    def __init__(self,input_dim,output_dim):
        super().__init__()
        self.linear=nn.Linear(input_dim,output_dim)
    def forward(self,x):
        x=self.linear(x)
        return x

## 9.导入数据

In [100]:
#原始的x_train给encoder输入，右移一位的y_train给decoder
data_path='./data/poem.txt'
data=pd.read_csv(data_path,header=None,names=['text'])
data['tokens']=data['text'].str.split()
#中文：data['tokens'] = data['text'].apply(lambda x: list(str(x)))

data.shape


(100, 2)

In [101]:

#tockenizer:
all_words=[word for tokens in data['tokens'] for word in tokens]
vocab={word:idx for idx,word in enumerate(set(all_words),start=1)}
vocab_size=len(set(all_words))
print('vocabulary:',vocab)
data['tokens_ids']=data['tokens'].apply(lambda x: [vocab[word] for word in x])

data.size

vocabulary: {'行': 1, '我': 2, '萄': 3, '阑': 4, '上': 5, '转': 6, '倚': 7, '乔': 8, '女': 9, '得': 10, '寻': 11, '使': 12, '碗': 13, '王': 14, '舍': 15, '如': 16, '竟': 17, '寺': 18, '衰': 19, '莺': 20, '京': 21, '画': 22, '浦': 23, '忆': 24, '魂': 25, '别': 26, '破': 27, '销': 28, '谢': 29, '色': 30, '蓬': 31, '荔': 32, '蹊': 33, '明': 34, '窗': 35, '一': 36, '风': 37, '满': 38, '洞': 39, '商': 40, '悲': 41, '面': 42, '鸭': 43, '菲': 44, '晚': 45, '芽': 46, '故': 47, '君': 48, '春': 49, '惘': 50, '琵': 51, '岸': 52, '寒': 53, '未': 54, '毛': 55, '掣': 56, '闺': 57, '近': 58, '哀': 59, '定': 60, '觉': 61, '逢': 62, '等': 63, '真': 64, '瀚': 65, '高': 66, '亲': 67, '指': 68, '钟': 69, '珠': 70, '罗': 71, '纸': 72, '独': 73, '甲': 74, '困': 75, '繁': 76, '扑': 77, '问': 78, '穿': 79, '空': 80, '葡': 81, '阴': 82, '柳': 83, '乘': 84, '悔': 85, '蚕': 86, '尖': 87, '安': 88, '却': 89, '海': 90, '烛': 91, '蒙': 92, '川': 93, '落': 94, '古': 95, '怜': 96, '北': 97, '香': 98, '难': 99, '秦': 100, '客': 101, '断': 102, '鸣': 103, '婿': 104, '饮': 105, '何': 106, '苦': 107, '红': 108, '栋': 109, '路': 

300

In [102]:
input_ids=np.array(data['tokens_ids'].tolist())

batch_size,_=input_ids.shape
input_ids.shape,batch_size


((100, 15), 100)

In [103]:
x=torch.tensor(input_ids)
x.shape


torch.Size([100, 15])

In [104]:
#数据准备
x_train=x[:,:-1]#给encoder
y_train=x[:,1:] #给decoder
print("x_train shape:", x_train.shape)  
print("y_train shape:", y_train.shape)
x_train.dtype

x_train shape: torch.Size([100, 14])
y_train shape: torch.Size([100, 14])


torch.int64

## 10.输入处理集成

In [105]:
pos=x_train.shape[1]
#数据预处理：embedding与positionalembedding
class Input_operation(nn.Module):
    def __init__(self,num_embeddings=vocab_size+1,d_model=d_model,max_len=pos):
        super().__init__()
        self.embed_model=Embedding(num_embeddings=vocab_size+1,d_model=d_model)
        self.positionalembed_model=PositionalEmbed(d_model,max_len)
    def forward(self,x):
        pos=x.shape[1]
        x_stand=self.positionalembed_model(self.embed_model(x))
    
        return x_stand
        

## 11.Transformer类封装、训练

In [106]:

class Transformer(nn.Module):
    def __init__(self,d_model,d_k,d_v,d_ff):
        super().__init__()
        self.decoder=Decoder(d_model,d_k,d_v,d_ff)

    def forward(self,x):
        y_h=self.decoder(x)#将输出的x状态再输入
        return y_h


model=Transformer(d_model,d_k,d_v,d_ff)
model.to(device)



input_operation=Input_operation(vocab_size+1,d_model)
output_operation=Output_operation(d_model,vocab_size+1)

input_operation=input_operation.to(device)
output_operation=output_operation.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(list(model.parameters())+list(input_operation.parameters())+list(output_operation.parameters()),lr=lr)


for epoch in tqdm(range(epochs)):
    x_train=x_train.to(device)
    y_train=y_train.to(device)
    #y预处理：embed,positionembed
    x_in=input_operation(x_train)

    y_pred=model(x_in)

    y_pred=output_operation(y_pred)
    y_pred=y_pred.transpose(1,2)
    y_train=y_train.squeeze(-1)
    loss=criterion(y_pred, y_train) 
    # 3. 反向传播 + 更新参数
    optimizer.zero_grad()   # 清空上一步的梯度
    loss.backward()         # 计算梯度
    optimizer.step()        # 更新参数
    
    if (epoch + 1) % 100 == 0:
        print(f"epoch {epoch+1}, loss: {loss.item():.4f}")

 10%|▉         | 100/1001 [00:13<02:17,  6.53it/s]

epoch 100, loss: 0.1794


 20%|█▉        | 200/1001 [00:27<02:07,  6.29it/s]

epoch 200, loss: 0.0545


 30%|██▉       | 300/1001 [00:40<01:49,  6.39it/s]

...

epoch 800, loss: 0.0182


 90%|████████▉ | 900/1001 [02:02<00:15,  6.42it/s]

epoch 900, loss: 0.0167


100%|██████████| 1001/1001 [02:16<00:00,  7.35it/s]

epoch 1000, loss: 0.0159


In [111]:
import torch
import math

def interactive_test(model, input_operation, output_operation, vocab, device):
    """
    交互式测试函数：
    1. 自动进行形状测试（非交互）
    2. 让用户输入起始词和生成长度，进行文本生成
    """
    print("="*60)
    print("交互式测试启动")
    
    # ---- 1. 检查变量 ----
    required_vars = ['model', 'input_operation', 'output_operation', 'vocab', 'device']
    missing = [v for v in required_vars if v not in globals()]
    if missing:
        print(f"❌ 缺少变量: {missing}，请先运行训练代码。")
        return

    # ---- 2. 自动形状测试（保持原样） ----
    model.eval()
    model.to(device)
    input_operation.to(device)
    output_operation.to(device)
    
    print("\n--- 自动形状测试 ---")
    with torch.no_grad():
        dummy = torch.randint(1, len(vocab)+1, (2, 10)).to(device)
        x_emb = input_operation(dummy)
        out = model(x_emb)
        logits = output_operation(out)
        print(f"✅ 形状测试通过: logits shape = {logits.shape} (预期 (2,10,{len(vocab)+1}))")
    print("-"*60)

    # ---- 3. 交互式生成 ----
    while 1:
        print("\n--- 文本生成交互 ---")
        print("请输入起始词（必须存在于词表中），或直接按 Enter 使用默认词（'铜'）")
    
        # 获取起始词
        start_word = input("起始词: ").strip()
        if not start_word:
            start_word = '铜'   # 默认词
            print(f"使用默认起始词: '{start_word}'")
    
        # 检查词是否存在
        if start_word not in vocab:
            print(f"⚠️ 警告: '{start_word}' 不在词表中，将使用第一个词 '{list(vocab.keys())[114]}' 代替")
            start_id = list(vocab.values())[114]
            start_word = list(vocab.keys())[114]
        else:
            start_id = vocab[start_word]
    
        # 获取生成长度
        max_len_str = input("生成长度（默认14）: ").strip()
        if max_len_str.isdigit():
            max_len = int(max_len_str)
        else:
            max_len = 14
            print("使用默认长度 14")
    
        print(f"\n开始生成，起始词: '{start_word}'，总长度: {max_len}\n")
    
        # 生成过程
        generated = [start_id]
        input_seq = torch.tensor([generated], device=device)
    
        # 为了显示逐步结果，我们每一步都打印当前生成的词
        id2word = {v: k for k, v in vocab.items()}
    
        for step in range(max_len - 1):
            with torch.no_grad():
                x_emb = input_operation(input_seq)
                out = model(x_emb)
                logits = output_operation(out)
                next_logits = logits[0, -1, :]   # 取最后一个位置
                next_token = torch.argmax(next_logits).item()
                generated.append(next_token)
                # 更新输入序列
                input_seq = torch.tensor([generated], device=device)
        
            # 实时打印生成的词
            word = id2word.get(next_token, '<unk>')
            print(f"步骤 {step+1}: 生成 '{word}'")
    
        # 最终结果
        words = [id2word.get(id, '<unk>') for id in generated]
        print("\n" + "="*60)
        print("最终生成的序列:")
        print(" ".join(words))
        print("="*60)
        
        model.train()
        print("交互测试结束。")

# 调用交互测试
interactive_test(model, input_operation, output_operation, vocab, device)

交互式测试启动

--- 自动形状测试 ---
✅ 形状测试通过: logits shape = torch.Size([2, 10, 615]) (预期 (2,10,615))
------------------------------------------------------------

--- 文本生成交互 ---
请输入起始词（必须存在于词表中），或直接按 Enter 使用默认词（'铜'）


起始词:  


使用默认起始词: '铜'


生成长度（默认14）:  


使用默认长度 14

开始生成，起始词: '铜'，总长度: 14

步骤 1: 生成 '雀'
步骤 2: 生成 '桥'
步骤 3: 生成 '边'
步骤 4: 生成 '野'
步骤 5: 生成 '草'
步骤 6: 生成 '花'
步骤 7: 生成 '乌'
步骤 8: 生成 '衣'
步骤 9: 生成 '巷'
步骤 10: 生成 '口'
步骤 11: 生成 '夕'
步骤 12: 生成 '阳'
步骤 13: 生成 '斜'

最终生成的序列:
铜 雀 桥 边 野 草 花 乌 衣 巷 口 夕 阳 斜
交互测试结束。

--- 文本生成交互 ---
请输入起始词（必须存在于词表中），或直接按 Enter 使用默认词（'铜'）


KeyboardInterrupt: Interrupted by user